# Eksploracyjna Analiza Danych (EDA)
## Predykcja podium (TOP 3) w wyścigach Formuły 1

Notebook zawiera eksploracyjną analizę zbioru danych F1 przed etapem modelowania.
Wykresy są zapisywane do folderu `Plots/`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('Plots', exist_ok=True)

df = pd.read_csv(os.path.join('f1_data_cleaned', 'f1_processed_dataset.csv'))

numerical_features = ['grid', 'year', 'round', 'circuitId', 'driver_age', 'quali_position']
categorical_features = ['driver_nationality', 'constructor_nationality']

print(f'Ksztalt zbioru: {df.shape}')
print(f'Kolumny: {list(df.columns)}')


In [ ]:
print('Rozkład zmiennej docelowej top3:')
print(df['top3'].value_counts())
print(f'Udzial klasy pozytywnej (podium): {df["top3"].mean()*100:.2f}%')
print()
print('Statystyki opisowe cech numerycznych:')
display(df[numerical_features + ['top3']].describe().round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['top3'].value_counts().sort_index()
axes[0].bar(['Brak podium (0)', 'Podium (1)'], counts.values,
            color=['#4C72B0', '#DD8452'], edgecolor='black')
axes[0].set_title('Rysunek 1. Rozklad zmiennej docelowej top3')
axes[0].set_ylabel('Liczba obserwacji')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['Brak podium (0)', 'Podium (1)'],
            autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Rysunek 1b. Proporcje klas w zbiorze danych')

plt.tight_layout()
plt.savefig('Plots/eda_rozklad_klas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Zapisano: Plots/eda_rozklad_klas.png')


In [ ]:
corr_matrix = df[numerical_features + ['top3']].corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Rysunek 2. Macierz korelacji cech numerycznych ze zmienna docelowa top3', fontsize=12)
plt.tight_layout()
plt.savefig('Plots/eda_macierz_korelacji.png', dpi=150, bbox_inches='tight')
plt.show()
print('Zapisano: Plots/eda_macierz_korelacji.png')


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

features_to_plot = ['grid', 'quali_position', 'driver_age', 'year', 'round', 'circuitId']
labels = ['Pozycja startowa (grid)', 'Pozycja kwalifikacyjna', 'Wiek kierowcy',
          'Rok wyscigu', 'Numer rundy', 'ID toru']

for i, (feat, label) in enumerate(zip(features_to_plot, labels)):
    df.boxplot(column=feat, by='top3', ax=axes[i],
               boxprops=dict(color='#4C72B0'),
               medianprops=dict(color='red', linewidth=2),
               whiskerprops=dict(color='#4C72B0'),
               capprops=dict(color='#4C72B0'),
               flierprops=dict(marker='o', color='gray', alpha=0.3, markersize=3))
    axes[i].set_title(label, fontsize=10)
    axes[i].set_xlabel('top3 (0 = brak podium, 1 = podium)')
    axes[i].set_ylabel(feat)

plt.suptitle('Rysunek 3. Boxploty cech numerycznych wedlug zmiennej docelowej top3', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('Plots/eda_boxploty.png', dpi=150, bbox_inches='tight')
plt.show()
print('Zapisano: Plots/eda_boxploty.png')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, grp in df.groupby('top3'):
    lbl = 'Podium (1)' if label == 1 else 'Brak podium (0)'
    axes[0].hist(grp['grid'], bins=30, alpha=0.6, label=lbl, density=True)
    axes[1].hist(grp['quali_position'], bins=30, alpha=0.6, label=lbl, density=True)

axes[0].set_title('Rysunek 4. Rozklad pozycji startowej (grid) wg klasy')
axes[0].set_xlabel('Pozycja startowa')
axes[0].set_ylabel('Gestosc')
axes[0].legend()

axes[1].set_title('Rysunek 4b. Rozklad pozycji kwalifikacyjnej wg klasy')
axes[1].set_xlabel('Pozycja kwalifikacyjna')
axes[1].set_ylabel('Gestosc')
axes[1].legend()

plt.tight_layout()
plt.savefig('Plots/eda_histogramy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Zapisano: Plots/eda_histogramy.png')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col, title in zip(axes,
                            ['driver_nationality', 'constructor_nationality'],
                            ['Rysunek 5. TOP 15 narodowosci kierowcow na podium',
                             'Rysunek 5b. TOP 15 narodowosci konstruktorow na podium']):
    top = (df[df['top3'] == 1][col]
           .value_counts()
           .head(15))
    ax.barh(top.index[::-1], top.values[::-1], color='#4C72B0', edgecolor='black')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Liczba podiow')

plt.tight_layout()
plt.savefig('Plots/eda_narodowosci.png', dpi=150, bbox_inches='tight')
plt.show()
print('Zapisano: Plots/eda_narodowosci.png')
